In [ ]:
%load_ext autoreload
%autoreload 2
from dataclasses import replace
from rankers.config import Config
from rankers.simulate import run_replicates, run
import numpy as np
import matplotlib.pyplot as plt

from rankers.metrics import neighbor_homophily_series

# Shared base: fixed-magnitude symmetric pool, baseline updating,
# neighbor reception, grow-from-seed repertoire, window of 1.
base = Config(
    n=500,
    k=32,
    p_rewire=0.01,

    n_claims=200,
    claim_scheme="fixed",
    llr_mag=1.0,

    repertoire_seed_size=5,
    belief_std=0.5,

    history_window=1,
    n_surfaced=1,
    ranker="baseline",
    receiver="neighbors",

    biases=("baseline",),

    n_steps=8000,
    record_every=1,
    n_tracked=50,
    seed=42,
)
def seen_series(result):
    return result.repertoire_history.mean(axis=1)        # unique claims seen (after > 0 fix)

def polarization_series(result):
    p = 1.0 / (1.0 + np.exp(-result.full_belief_traj))
    return 4.0 * p.var(axis=1)     

In [ ]:
beta = 512
n_runs = 1
receivers = ["neighbors", "external"]

results_by_recv = {}
for recv in receivers:
    cfg_recv = replace(base, emission_temp=beta, receiver=recv)
    results_by_recv[recv] = [run(replace(cfg_recv, seed=base.seed + r)) for r in range(n_runs)]

pol_by_recv  = {r: np.stack([polarization_series(res) for res in rs]) for r, rs in results_by_recv.items()}
seen_by_recv = {r: np.stack([seen_series(res) for res in rs]) for r, rs in results_by_recv.items()}
homo_by_recv = {r: np.stack([neighbor_homophily_series(res) for res in rs]) for r, rs in results_by_recv.items()}


In [ ]:
steps = np.arange(next(iter(pol_by_recv.values())).shape[1]) * base.record_every
recv_colors = {"neighbors": "tab:red", "external": "tab:blue"}

fig, (ax_pol, ax_seen, ax_homo) = plt.subplots(1, 3, figsize=(20, 5), sharex=True)

for recv in receivers:
    c = recv_colors[recv]
    ax_pol.plot(steps,  pol_by_recv[recv].mean(axis=0),  color=c, lw=1.5, label=recv)
    ax_seen.plot(steps, seen_by_recv[recv].mean(axis=0), color=c, lw=1.5, label=recv)
    ax_homo.plot(steps, homo_by_recv[recv].mean(axis=0), color=c, lw=1.5, label=recv)

ax_seen.axhline(base.n_claims, color="black", ls="--", lw=0.8, label=f"full pool ({base.n_claims})")
ax_homo.axhline(0.0, color="black", ls="--", lw=0.8)

ax_pol.set_ylabel("polarization  (4·Var of σ(belief))"); ax_pol.set_ylim(0, 1); ax_pol.set_title(f"Polarization (β = {beta})")
ax_seen.set_ylabel("avg unique claims seen"); ax_seen.set_ylim(0, base.n_claims * 1.05); ax_seen.set_title("Repertoire saturation")
ax_homo.set_ylabel("edge homophily"); 
#ax_homo.set_ylim(-0.1, 1.0); 
ax_homo.set_title("Network homophily")

for ax in (ax_pol, ax_seen, ax_homo):
    ax.set_xlabel("step"); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
def polarization(beliefs):
    p = 1.0 / (1.0 + np.exp(-beliefs))
    return 4.0 * p.var()

In [ ]:
fig, (ax_n, ax_e) = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, recv, color in [(ax_n, "neighbors", "tab:red"), (ax_e, "external", "tab:blue")]:
    finals = np.concatenate([res.final_beliefs for res in results_by_recv[recv]])
    ax.hist(finals, bins=60, color=color, alpha=0.7)
    ax.axvline(0, color="black", lw=0.8, ls="--")
    beta_run = results_by_recv[recv][0].cfg.emission_temp
    ax.set_title(f"{recv}  (β = {beta_run:.0f}, pol = {polarization(finals):.3f})")
    ax.set_xlabel("final belief (log-odds)")

plt.tight_layout()
plt.show()

In [ ]:
def plot_homophily_diagnostic(result, ax=None):
    """Neighbor vs random pairwise belief distance over time; gap = crystallization."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))

    neighbor_dist, random_dist, gap = neighbor_homophily_series(result)
    steps = np.arange(len(gap)) * result.cfg.record_every

    ax.plot(steps, random_dist,   color="tab:gray", lw=1.5, label="random pairs")
    ax.plot(steps, neighbor_dist, color="tab:red",  lw=1.5, label="graph neighbors")
    ax.fill_between(steps, neighbor_dist, random_dist, color="tab:red", alpha=0.15,
                    label="gap (homophily)")

    ax.set_xlabel("step")
    ax.set_ylabel("mean |l_i − l_j|  (log-odds)")
    ax.set_title(f"Crystallization diagnostic (β = {result.cfg.emission_temp:.0f}, "
                 f"receiver = {result.cfg.receiver})")
    ax.legend()
    return ax

In [ ]:
plot_homophily_diagnostic(results_by_recv["neighbors"][0])
plt.show()